# Дообучение Vikhr-Qwen-2.5-1.5B-Instruct под реплики NPC Skyrim (Mantella)
QLoRA → merge → GGUF Q8_0. Запускать в Colab (Runtime → T4 GPU) или Kaggle (GPU T4).
**Перед запуском:** загрузите `dataset.jsonl` в файлы Colab (панель слева).

In [ ]:
!pip install -q transformers peft trl accelerate bitsandbytes datasets sentencepiece

In [ ]:
import torch, json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

BASE = "Vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct"   # оригинальные веса, НЕ gguf
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

ds = load_dataset("json", data_files="dataset_swear.jsonl", split="train")
ds = ds.train_test_split(test_size=0.08, seed=42)

def fmt(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}
ds = ds.map(fmt, remove_columns=ds["train"].column_names)
print(ds["train"][0]["text"][-600:])

In [ ]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto", dtype=torch.float16)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = SFTConfig(
    output_dir="out", num_train_epochs=3, per_device_train_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_steps=3, logging_steps=5, eval_strategy="epoch", save_strategy="no",
    fp16=True, max_length=1536, dataset_text_field="text", report_to="none",
)
from peft import prepare_model_for_kbit_training, get_peft_model
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora)
for p in model.parameters():          # обучаемые веса LoRA -> fp32, иначе fp16-GradScaler падает
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds["train"],
                     eval_dataset=ds["test"], processing_class=tok)
trainer.train()
trainer.model.save_pretrained("lora_npc")

## Быстрый тест (до конвертации)

In [ ]:
model.eval()
msgs = [{"role":"system","content":"You are Ралоф in Skyrim. Output ONLY spoken dialogue. Respond as Ралоф now in Russian."},
        {"role":"user","content":"Привет, Ралоф. Как дела?"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## Merge LoRA в полные веса (fp16) и конвертация в GGUF

In [ ]:
import gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, "lora_npc").merge_and_unload()
merged.save_pretrained("merged"); tok.save_pretrained("merged")

In [ ]:
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py merged --outfile vikhr-npc-q8_0.gguf --outtype q8_0
!ls -lh vikhr-npc-q8_0.gguf

In [ ]:
from google.colab import files   # в Kaggle файл лежит в /kaggle/working
files.download("vikhr-npc-q8_0.gguf")

## Дальше
1. Положите `vikhr-npc-q8_0.gguf` рядом со старой моделью, запустите KoboldCpp с ним.
2. В Mantella: LLM → Custom Token Count поставьте реальный контекст модели (Vikhr-Qwen 2.5: до 32768, но в KoboldCpp задайте `--contextsize 8192`).
3. Генерация: temperature 0.6–0.8, `stop: ["#"]` уже стоит в llm_params.